In [4]:
# ============================================================================
# 01_generation.ipynb  —  Rating + rationale generation (gap-given / gap-blind)
# ----------------------------------------------------------------------------
# Single-cell notebook. All code and comments are in English.
# Figures: seaborn grayscale, no captions, dpi=600, saved as both PNG and PDF.
# Run order: 00 -> 01 -> 02 -> 03 -> 04 -> 05.
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Make src/ importable regardless of the notebook's working directory.
_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
                    load_env, set_grayscale_style, save_fig, save_table,
                    require_input, RANDOM_SEED)

import matplotlib.pyplot as plt
import seaborn as sns
set_grayscale_style()
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")
# NOTE: absolute local paths are never printed (anonymity for double-blind review).

# ----------------------------------------------------------------------------
# PURPOSE
#   For each override case, generate a rating and a free-text rationale with the
#   LLM generator under two conditions for the circularity ablation:
#     GAP-GIVEN  : the generator is told the notch gap and asked to articulate
#                  the matching qualitative factor.
#     GAP-BLIND  : the generator is NOT told the gap.
#   NEW vs. prior study: a REAL-MDNA condition can ground the rationale in the
#   firm's actual 10-K MD&A text instead of a model-invented justification.
#
# INPUT   data/with_override.parquet
# OUTPUT  data/generated.parquet  (case_id, condition, rating_gen, rationale)
#
# NOTE    Requires OPENAI_API_KEY in .env. Uses the WEAK/MID/STRONG model tiers.
#         Set MAX_CASES small first to smoke-test cost, then scale up.
# ----------------------------------------------------------------------------
cfg = load_env()
from openai import OpenAI
client = OpenAI(api_key=cfg["api_key"])

MODEL = cfg["mid"]          # generator tier; swap to cfg["strong"]/cfg["weak"] as needed
MAX_CASES = 20              # <-- raise after a cost check
TEMPERATURE = 0.2

ov = pd.read_parquet(require_input(DATA_DIR / "with_override.parquet",
                                   hint="Run 00_proxy_override first."))
ov = ov[ov["override"]].copy().head(MAX_CASES)

def gen_one(row, condition):
    gap_txt = (f"The agency rating sits {int(row.delta):+d} notches versus the "
               f"quantitative stand-alone profile." if condition == "gap_given"
               else "")
    mdna = (str(row.get("mdna_text", ""))[:6000])  # truncate for context budget
    sys_msg = ("You are a credit analyst. Produce a letter rating and a short "
               "rationale citing quantitative and qualitative factors (parent/"
               "group/government support, industry, governance).")
    user_msg = (f"Financial sector: {row.get('Sector','NA')}.\n"
                f"{gap_txt}\n"
                f"Relevant MD&A excerpt:\n{mdna}\n\n"
                f"Return JSON: {{\"rating\": <letter>, \"rationale\": <text>}}")
    r = client.chat.completions.create(
        model=MODEL, temperature=TEMPERATURE,
        messages=[{"role":"system","content":sys_msg},
                  {"role":"user","content":user_msg}],
        response_format={"type":"json_object"})
    return json.loads(r.choices[0].message.content)

rows = []
for condition in ("gap_given", "gap_blind"):
    for _, row in ov.iterrows():
        try:
            out = gen_one(row, condition)
        except Exception as e:
            out = {"rating": None, "rationale": f"[gen-error] {e}"}
        rows.append({"case_id": f"{row.cik}_{row.rating_year}",
                     "cik": row.cik, "rating_year": int(row.rating_year),
                     "delta": int(row.delta), "condition": condition,
                     "rating_gen": out.get("rating"),
                     "rationale": out.get("rationale")})

gen = pd.DataFrame(rows)
gen.to_parquet(DATA_DIR / "generated.parquet", index=False)
print(gen.groupby("condition").size())
print(gen.head(3).to_string())


condition
gap_blind    20
gap_given    20
dtype: int64
       case_id     cik  rating_year  delta  condition rating_gen                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 